# 🚀 Google Colab Live AI Video API Server (Zero Signup - Cloudflare Tunnel)
Bu notebook, **Cloudflare Tunnel (trycloudflare.com)** kullanır. Hesap oluşturma veya ngrok token gerektirmez, %100 ücretsiz canlı HTTPS API adresi üretir.

In [ ]:
# 1. Gerekli Kütüphaneler ve Cloudflare Tunnel Kurulumu
!pip install -q diffusers transformers accelerate torch torchvision imageio-ffmpeg fastapi uvicorn hf_transfer
!wget -q -O cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared.deb
import os, torch
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Kurulum Tamamlandı! Kullanılan Donanım: {device.upper()}')

In [ ]:
# 2. Text-to-Video AI Modelinin Yüklenmesi
from diffusers import DiffusionPipeline
from diffusers.utils import export_to_video

print(f'🚀 AI Video Modeli {device.upper()} Üzerinde Yükleniyor...')
dtype = torch.float16 if device == 'cuda' else torch.float32
pipe = DiffusionPipeline.from_pretrained(
    'damo-vilab/text-to-video-ms-1.7b',
    torch_dtype=dtype
)
pipe = pipe.to(device)
if device == 'cuda':
    pipe.enable_attention_slicing()
print('✅ AI Video Modeli Başarıyla Yüklendi!')

In [ ]:
# 3. Canlı Cloudflare Tunnel + FastAPI Web Sunucusu
import subprocess, time
from fastapi import FastAPI, Response
from pydantic import BaseModel
import uvicorn
import nest_asyncio

app = FastAPI()

class VideoRequest(BaseModel):
    prompt: str
    niche: str = 'minecraft'
    width: int = 256
    height: int = 448

@app.get('/')
def health_check():
    return {'status': 'online', 'model': 'ModelScope 1.7B', 'device': device}

@app.post('/generate_video')
def generate_video(req: VideoRequest):
    print(f'🎬 Otomasyondan canlı video isteği alındı: {req.prompt}')
    video_frames = pipe(
        prompt=req.prompt,
        num_inference_steps=20,
        height=448,
        width=256,
        num_frames=16
    ).frames[0]
    
    out_path = '/content/colab_generated_video.mp4'
    export_to_video(video_frames, out_path, fps=16)
    
    with open(out_path, 'rb') as f:
        return Response(content=f.read(), media_type='video/mp4')

# Start Cloudflare Tunnel in background
subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--logfile', '/content/cloudflared.log'])
time.sleep(4)

print('====================================================')
print('🚀 CANLI GOOGLE COLAB API URL ADRESİNİZ (Cloudflare):')
try:
    with open('/content/cloudflared.log') as f:
        for line in f:
            if 'trycloudflare.com' in line:
                url = [x for x in line.split() if 'trycloudflare.com' in x][0]
                print(url)
                print(f'COLAB_API_URL={url}')
except Exception:
    pass
print('====================================================')

nest_asyncio.apply()
uvicorn.run(app, host='0.0.0.0', port=8000)